# Важно: я не починил этот ноутбук, он не рабочий
TODO: починить этот ноутбук

In [1]:
import sys, os, json, torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
warnings = __import__('warnings')
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('../../..'))

In [2]:
from DL.models.custom_classifier_from_config import CustomClassifierFromConfig
from DL.trainers.model_trainer import ModelTrainer
from DL.visualization.plot_utils import plot_confusion_matrices
from DL.data.synthetic_cyrillic_dataset import SyntheticCyrillicDataset

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device: ', device)

device:  cuda


In [4]:
perspective_transform = T.RandomPerspective(distortion_scale=0.5, p=0.8)

train_dataset = SyntheticCyrillicDataset(60000, 42, transform=perspective_transform)
test_dataset = SyntheticCyrillicDataset(15000, 999, transform=perspective_transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [5]:
def run_advanced(name, config_kwargs, trainer_kwargs):
    with open('../../../DL/configs/lenet/lenet_4kb.json', 'r') as f:
        config = json.load(f)
    config['num_classes'] = len(train_dataset.char_to_idx)

    model = CustomClassifierFromConfig(config, **config_kwargs).to(device)
    trainer = ModelTrainer(model, torch.optim.Adam(model.parameters(), lr=1e-3), nn.CrossEntropyLoss(), device, **trainer_kwargs)
    print(f"\n=== {name} ===")
    trainer.fit(train_loader, test_loader, epochs=10)
    plot_confusion_matrices(model, model, test_loader, test_dataset, device)

In [ ]:
# 1. Pixel Count
run_advanced("Feature Eng (Pixel Count)", {'use_pixel_count': True}, {'use_background_loss': True, 'bg_loss_alpha': 15.0, 'bg_loss_n': 2})


=== Feature Eng (Pixel Count) ===
Эпоха 1/10


Трейн Лосс: 2.5626 | Точность: 0.3693 | BG Лосс: 0.1280
Валид Лосс: 1.7580 | Точность: 0.5938

Эпоха 2/10


Трейн Лосс: 1.3839 | Точность: 0.6666 | BG Лосс: 0.0142
Валид Лосс: 1.1022 | Точность: 0.7219

Эпоха 3/10


Трейн Лосс: 0.9571 | Точность: 0.7566 | BG Лосс: 0.0164
Валид Лосс: 0.8407 | Точность: 0.7853

Эпоха 4/10


Трейн Лосс: nan | Точность: 0.5804 | BG Лосс: nan
Валид Лосс: nan | Точность: 0.0298

Эпоха 5/10


Трейн Лосс: nan | Точность: 0.0289 | BG Лосс: nan
Валид Лосс: nan | Точность: 0.0319

Эпоха 6/10


Трейн Лосс: nan | Точность: 0.0299 | BG Лосс: nan
Валид Лосс: nan | Точность: 0.0289

Эпоха 7/10


Трейн Лосс: nan | Точность: 0.0299 | BG Лосс: nan
Валид Лосс: nan | Точность: 0.0311

Эпоха 8/10


Обучение:  81%|████████▏ | 191/235 [00:36<00:08,  5.26it/s, loss=nan]

In [ ]:
# 2. Metric Learning (ArcFace)
run_advanced("Metric Learning (ArcFace)", {'use_arcface': True}, {'use_background_loss': True, 'bg_loss_alpha': 15.0, 'bg_loss_n': 2})

In [ ]:
# 3. Laten GAN
run_advanced("Latent GAN", {}, {'use_background_loss': True, 'bg_loss_alpha': 15.0, 'bg_loss_n': 2, 'use_gan_loss': True})